# Projeto (Seção 9) — solução comentada

Continua a solução comentada da seção 8. Enunciado em [`04_projeto`](04_projeto.ipynb).

Requisitos do enunciado + dois cuidados que a solução do instrutor não toma:
- o campo calculado precisa aceitar **país ausente** (`str | None`), senão o `model_dump()` quebra;
- se o valor é **cacheado** e a origem é **mutável**, o cache precisa ser **invalidado** quando a origem muda.

## 1. Solução

In [1]:
from datetime import date
from enum import Enum
from functools import cached_property
from typing import Annotated, TypeVar
from uuid import UUID, uuid4
from pydantic import (UUID4, AfterValidator, BaseModel, ConfigDict, Field, ValidationError, ValidationInfo,
                      computed_field, field_serializer, field_validator)
from pydantic.alias_generators import to_camel

countries = {
    "australia": ("Australia", "AUS"), "canada": ("Canada", "CAN"), "china": ("China", "CHN"),
    "france": ("France", "FRA"), "germany": ("Germany", "DEU"), "india": ("India", "IND"),
    "mexico": ("Mexico", "MEX"), "norway": ("Norway", "NOR"), "pakistan": ("Pakistan", "PAK"),
    "san marino": ("San Marino", "SMR"), "sanmarino": ("San Marino", "SMR"), "spain": ("Spain", "ESP"),
    "sweden": ("Sweden", "SWE"), "united kingdom": ("United Kingdom", "GBR"), "uk": ("United Kingdom", "GBR"),
    "great britain": ("United Kingdom", "GBR"), "britain": ("United Kingdom", "GBR"),
    "us": ("United States of America", "USA"), "united states": ("United States of America", "USA"),
    "usa": ("United States of America", "USA"), "brazil": ("Brazil", "BRA"), "brasil": ("Brazil", "BRA"),
}
VALID_COUNTRY_KEYS = sorted(countries)
country_code_lookup = {name: code for name, code in countries.values()}

DB_CALLS = 0                                    # contador para provar o efeito do cache


def lookup_country(name: str) -> tuple[str, str]:
    key = " ".join(name.split()).casefold()
    try:
        return countries[key]
    except KeyError:
        raise ValueError(f"unknown country {name!r}; expected one of: {', '.join(VALID_COUNTRY_KEYS)}") from None


def fetch_country_code(country_name: str) -> str:
    """Simula a consulta ao banco (é isso que justifica o cache)."""
    global DB_CALLS
    DB_CALLS += 1
    return country_code_lookup[country_name]


class AutomobileType(Enum):
    sedan = "Sedan"
    coupe = "Coupe"
    convertible = "Convertible"
    suv = "SUV"
    truck = "Truck"


T = TypeVar("T")
BoundedString = Annotated[str, Field(min_length=2, max_length=50)]
BoundedList = Annotated[list[T], Field(min_length=1, max_length=5)]
Country = Annotated[str, AfterValidator(lambda name: lookup_country(name)[0])]


def make_alias(field_name: str) -> str:
    return to_camel(field_name).removesuffix("_")


class Automobile(BaseModel):
    model_config = ConfigDict(
        extra="forbid",
        str_strip_whitespace=True,
        validate_default=True,
        validate_assignment=True,
        alias_generator=make_alias,
    )

    id_: UUID4 = Field(default_factory=uuid4)
    manufacturer: BoundedString
    series_name: BoundedString
    type_: AutomobileType
    is_electric: bool = Field(default=False, repr=False)
    manufactured_date: date = Field(validation_alias="completionDate", ge=date(1980, 1, 1), repr=False)
    base_msrp_usd: float = Field(validation_alias="msrpUSD", serialization_alias="baseMSRPUSD", repr=False)
    top_features: BoundedList[BoundedString] | None = Field(default=None, repr=False)
    vin: BoundedString = Field(repr=False)
    number_of_doors: int = Field(default=4, validation_alias="doors", ge=2, le=4, multiple_of=2, repr=False)
    registration_country: Country | None = Field(default=None, repr=False)
    registration_date: date | None = Field(default=None, repr=False)
    license_plate: BoundedString | None = Field(default=None, repr=False)

    @computed_field(repr=False)                    # alias registrationCountryCode vem do make_alias
    @cached_property
    def registration_country_code(self) -> str | None:
        if self.registration_country is None:
            return None
        return fetch_country_code(self.registration_country)

    def __setattr__(self, name, value):
        super().__setattr__(name, value)           # valida (validate_assignment) e grava
        if name == "registration_country":
            self.__dict__.pop("registration_country_code", None)   # invalida o cache

    @field_serializer("manufactured_date", "registration_date", when_used="json-unless-none")
    def serialize_date(self, value: date) -> str:
        return value.strftime("%Y/%m/%d")

    @field_validator("manufactured_date")
    @classmethod
    def manufactured_not_after_registration(cls, value: date, info: ValidationInfo) -> date:
        registered = info.data.get("registration_date")
        if registered is not None and value > registered:
            raise ValueError(f"manufactured_date {value} is after registration_date {registered}")
        return value

    @field_validator("registration_date")
    @classmethod
    def registration_not_before_manufacture(cls, value: date | None, info: ValidationInfo) -> date | None:
        manufactured = info.data.get("manufactured_date")
        if value is not None and manufactured is not None and value < manufactured:
            raise ValueError(f"registration_date {value} is before manufactured_date {manufactured}")
        return value

## 2. Teste do enunciado

In [2]:
data = {
    "id": "c4e60f4a-3c7f-4da5-9b3f-07aee50b23e7",
    "manufacturer": "BMW",
    "seriesName": "M4 Competition xDrive",
    "type": "Convertible",
    "isElectric": False,
    "completionDate": "2023-01-01",
    "msrpUSD": 93_300,
    "topFeatures": ["6 cylinders", "all-wheel drive", "convertible"],
    "vin": "1234567890",
    "doors": 2,
    "registrationCountry": "us",
    "registrationDate": "2023-06-01",
    "licensePlate": "AAA-BBB"
}

expected_serialized_by_alias = {
    'id': UUID('c4e60f4a-3c7f-4da5-9b3f-07aee50b23e7'),
    'manufacturer': 'BMW',
    'seriesName': 'M4 Competition xDrive',
    'type': AutomobileType.convertible,
    'isElectric': False,
    'manufacturedDate': date(2023, 1, 1),
    'baseMSRPUSD': 93300.0,
    'topFeatures': ['6 cylinders', 'all-wheel drive', 'convertible'],
    'vin': '1234567890',
    'numberOfDoors': 2,
    'registrationCountry': 'United States of America',
    'registrationCountryCode': 'USA',
    'registrationDate': date(2023, 6, 1),
    'licensePlate': 'AAA-BBB',
}

In [3]:
car = Automobile.model_validate(data)
assert car.model_dump(by_alias=True) == expected_serialized_by_alias
print(f"car: {car!r}")
print("teste do enunciado: ok")

car: Automobile(id_=UUID('c4e60f4a-3c7f-4da5-9b3f-07aee50b23e7'), manufacturer='BMW', series_name='M4 Competition xDrive', type_=<AutomobileType.convertible: 'Convertible'>)
teste do enunciado: ok


## 3. Testes extras

In [4]:
# repr limpo: só os 4 campos pedidos
assert repr(car).count("=") == 4

# cache: várias leituras/serializações, uma única "consulta ao banco"
DB_CALLS = 0
car = Automobile.model_validate(data)
for _ in range(5):
    car.registration_country_code
    car.model_dump()
print(f"consultas ao 'banco' para 5 leituras + 5 dumps: {DB_CALLS}")

# invalidação: trocar o país recalcula o código (uma nova consulta)
car.registration_country = "france"
print(f"país trocado: {car.registration_country} -> {car.registration_country_code} (consultas: {DB_CALLS})")
car.registration_country = None
print(f"país removido: {car.registration_country_code}")

# país ausente na entrada: serializa normalmente com código None
no_country = {k: v for k, v in data.items() if k != "registrationCountry"}
print(f"sem país: {Automobile.model_validate(no_country).model_dump(by_alias=True)['registrationCountryCode']}")

# código no JSON e no schema de serialização
import json
print(f"json['registrationCountryCode']: {json.loads(Automobile.model_validate(data).model_dump_json(by_alias=True))['registrationCountryCode']}")
print(f"no schema de serialização: {'registrationCountryCode' in Automobile.model_json_schema(mode='serialization')['properties']}")
print("testes extras: ok")

consultas ao 'banco' para 5 leituras + 5 dumps: 1
país trocado: France -> FRA (consultas: 2)
país removido: None
sem país: None
json['registrationCountryCode']: USA
no schema de serialização: True
testes extras: ok


## 4. Comparação com a solução do instrutor

| ponto | instrutor | esta solução |
|---|---|---|
| tipo do campo calculado | `str` | `str \| None` |
| país ausente | **`KeyError`** no `model_dump()` | `None` |
| cache × país mutável | **código desatualizado** após trocar o país | cache invalidado no `__setattr__` |
| `repr` limpo | `repr=False` campo a campo | idem |
| posição do campo calculado | declarado no meio dos campos | após os campos (campos calculados são serializados **depois** dos campos comuns, de qualquer forma) |
| prova do cache | — | contador de "consultas ao banco" |

> 💡 Alternativa válida ao `__setattr__`: **não cachear** (a consulta a um dict é barata) ou **congelar** `registration_country`
> (`Field(frozen=True)`). O cache com invalidação só se justifica quando a consulta é cara de verdade (banco, rede).